In [1]:
# ================================
# RAG HYPERPARAMETER GRID SEARCH
# With Advanced NLP Metrics
# ================================

# Step 1: Install required packages
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn
!pip install -q rouge-score nltk bert-score sacrebleu
!pip install -q sentence-transformers  # Ensure latest version for embeddings

# Step 2: Imports
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
import faiss
import numpy as np
from transformers import pipeline
from tqdm.auto import tqdm
import textwrap
import time
from datetime import datetime
from itertools import product
import warnings

warnings.filterwarnings("ignore")

# Metric imports
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Download NLTK data for METEOR
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ================================
# CONFIGURATION: Define Grid Search Space
# ================================

EXPERIMENT_CONFIG = {
    # Embedding models to test
    "embedding_models": [
        "BAAI/bge-small-en-v1.5",  # 384 dim, fast
        "sentence-transformers/all-MiniLM-L6-v2",  # 384 dim, popular
        "BAAI/bge-base-en-v1.5",  # 768 dim, better quality
    ],
    # Generation models to test
    "generation_models": [
        "Qwen/Qwen2.5-7B-Instruct",
        "mistralai/Mistral-7B-Instruct-v0.2",
        "google/flan-t5-base",  # Smaller, faster option
    ],
    # Retrieval parameters
    "top_k": [3, 5],
    # Generation hyperparameters
    "temperature": [0.3, 0.7],
    "top_p": [0.85, 0.95],
    "top_k_sampling": [30, 50],
    "max_new_tokens": [512],
}

# Test queries for evaluation with reference answers
TEST_QUERIES = [
    {
        "query": "Recommend a good fast charging USB-C cable under 300 rupees",
        "reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging.",
    },
    {
        "query": "Which cable has the highest rating and supports 60W charging?",
        "reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support.",
    },
    {
        "query": "What is the best iPhone lightning cable in the list?",
        "reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option.",
    },
    {
        "query": "Suggest me some good long lasting headphones",
        "reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379.",
    },
]
# Composite score weights
METRIC_WEIGHTS = {
    "rouge_1": 0.10,
    "rouge_l": 0.10,
    "bleu": 0.10,
    "meteor": 0.15,
    "bert_score": 0.25,
    "embedding_similarity": 0.20,
    "faithfulness": 0.10,
}

# ================================
# Step 3: Load Dataset
# ================================
print("Loading dataset...")
df = pd.read_csv("/kaggle/input/mlops-amazon/amazon.csv")
print(f"Loaded {len(df)} products")

# Create document corpus
documents = []
metadatas = []

for _, row in df.iterrows():
    text = f"""
Product: {row['product_name']}
Category: {row['category']}
Price: {row['discounted_price']} (was {row['actual_price']}, {row['discount_percentage']} off)
Rating: {row['rating']} ⭐ ({row['rating_count']} reviews)
Description: {row['about_product']}
Reviews: {row['review_content'][:1000]}...
""".strip()
    documents.append(text)
    metadatas.append(
        {
            "product_id": row["product_id"],
            "name": row["product_name"],
            "price": row["discounted_price"],
            "rating": row["rating"],
        }
    )

print(f"Created {len(documents)} searchable documents")

# ================================
# Step 4: Advanced Evaluation Metrics
# ================================


class AdvancedMetrics:
    """Comprehensive NLP evaluation metrics"""

    def __init__(self):
        # Initialize ROUGE scorer
        self.rouge_scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

        # Initialize BLEU
        self.bleu = BLEU()

        # Initialize embedding model for semantic similarity
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

        print("✓ Advanced metrics initialized")

    def calculate_rouge(self, prediction, reference):
        """Calculate ROUGE-1 and ROUGE-L scores"""
        scores = self.rouge_scorer.score(reference, prediction)
        return {
            "rouge_1_f1": scores["rouge1"].fmeasure,
            "rouge_1_precision": scores["rouge1"].precision,
            "rouge_1_recall": scores["rouge1"].recall,
            "rouge_l_f1": scores["rougeL"].fmeasure,
            "rouge_l_precision": scores["rougeL"].precision,
            "rouge_l_recall": scores["rougeL"].recall,
        }

    def calculate_bleu(self, prediction, reference):
        """Calculate BLEU score"""
        try:
            score = (
                self.bleu.sentence_score(prediction, [reference]).score / 100.0
            )  # Normalize to 0-1
            return score
        except:
            return 0.0

    def calculate_meteor(self, prediction, reference):
        """Calculate METEOR score"""
        try:
            pred_tokens = word_tokenize(prediction.lower())
            ref_tokens = word_tokenize(reference.lower())
            score = meteor_score([ref_tokens], pred_tokens)
            return score
        except:
            return 0.0

    def calculate_bert_score(self, predictions, references):
        """Calculate BERTScore (batch processing for efficiency)"""
        try:
            P, R, F1 = bert_score(
                predictions,
                references,
                lang="en",
                verbose=False,
                device="cuda" if torch.cuda.is_available() else "cpu",
            )
            return {
                "bert_score_f1": F1.mean().item(),
                "bert_score_precision": P.mean().item(),
                "bert_score_recall": R.mean().item(),
            }
        except Exception as e:
            print(f"BERTScore error: {e}")
            return {
                "bert_score_f1": 0.0,
                "bert_score_precision": 0.0,
                "bert_score_recall": 0.0,
            }

    def calculate_embedding_similarity(self, prediction, reference):
        """Calculate cosine similarity using sentence embeddings"""
        try:
            pred_emb = self.embed_model.encode(prediction, convert_to_tensor=True)
            ref_emb = self.embed_model.encode(reference, convert_to_tensor=True)
            similarity = util.cos_sim(pred_emb, ref_emb).item()
            return max(0.0, similarity)  # Ensure non-negative
        except:
            return 0.0

    def calculate_faithfulness(self, answer, context):
        """Check if answer is grounded in context using embeddings"""
        try:
            answer_emb = self.embed_model.encode(answer, convert_to_tensor=True)
            context_emb = self.embed_model.encode(context, convert_to_tensor=True)
            similarity = util.cos_sim(answer_emb, context_emb).item()

            # Additional heuristics
            has_specific_info = any(
                word in answer.lower() for word in ["rupees", "rating", "cable", "product", "brand"]
            )
            length_penalty = 1.0 if len(answer.split()) >= 10 else 0.7

            faithfulness = (similarity * 0.7) + (0.3 if has_specific_info else 0.0)
            faithfulness *= length_penalty

            return min(1.0, faithfulness)
        except:
            return 0.5

    def calculate_all_metrics(self, prediction, reference, context):
        """Calculate all metrics for a single prediction"""
        metrics = {}

        # ROUGE scores
        rouge_scores = self.calculate_rouge(prediction, reference)
        metrics.update(rouge_scores)

        # BLEU score
        metrics["bleu"] = self.calculate_bleu(prediction, reference)

        # METEOR score
        metrics["meteor"] = self.calculate_meteor(prediction, reference)

        # BERTScore (single item)
        bert_scores = self.calculate_bert_score([prediction], [reference])
        metrics.update(bert_scores)

        # Embedding similarity
        metrics["embedding_similarity"] = self.calculate_embedding_similarity(prediction, reference)

        # Faithfulness (context grounding)
        metrics["faithfulness"] = self.calculate_faithfulness(prediction, context)

        return metrics

    def calculate_composite_score(self, metrics):
        """Calculate weighted composite score"""
        composite = (
            metrics["rouge_1_f1"] * METRIC_WEIGHTS["rouge_1"]
            + metrics["rouge_l_f1"] * METRIC_WEIGHTS["rouge_l"]
            + metrics["bleu"] * METRIC_WEIGHTS["bleu"]
            + metrics["meteor"] * METRIC_WEIGHTS["meteor"]
            + metrics["bert_score_f1"] * METRIC_WEIGHTS["bert_score"]
            + metrics["embedding_similarity"] * METRIC_WEIGHTS["embedding_similarity"]
            + metrics["faithfulness"] * METRIC_WEIGHTS["faithfulness"]
        )
        return composite


# Initialize metrics calculator
metrics_calculator = AdvancedMetrics()

# ================================
# Step 5: RAG Pipeline with Configurable Parameters
# ================================


class ConfigurableRAG:
    def __init__(self, embedding_model_name, generation_model_name):
        self.embedding_model_name = embedding_model_name
        self.generation_model_name = generation_model_name
        self.embedder = None
        self.generator = None
        self.index = None
        self.embeddings = None

    def setup_embedder(self):
        """Load embedding model and create FAISS index"""
        print(f"Loading embedder: {self.embedding_model_name}")
        self.embedder = SentenceTransformer(self.embedding_model_name)

        # Get embedding dimension
        test_emb = self.embedder.encode(["test"], normalize_embeddings=True)
        dimension = test_emb.shape[1]

        # Create FAISS index
        self.index = faiss.IndexFlatIP(dimension)

        # Embed all documents
        batch_size = 32
        embeddings = []
        for i in tqdm(range(0, len(documents), batch_size), desc="Embedding docs"):
            batch = documents[i : i + batch_size]
            batch_emb = self.embedder.encode(batch, normalize_embeddings=True)
            embeddings.append(batch_emb)
            self.index.add(batch_emb)

        self.embeddings = np.vstack(embeddings)
        print(f"Indexed {self.index.ntotal} products")

    def setup_generator(self):
        """Load generation model"""
        print(f"Loading generator: {self.generation_model_name}")
        try:
            self.generator = pipeline(
                "text-generation",
                model=self.generation_model_name,
                torch_dtype=torch.bfloat16,
                device_map="auto",
            )
        except Exception as e:
            print(f"Error loading {self.generation_model_name}: {e}")
            self.generator = None

    def retrieve(self, query, top_k=5):
        """Retrieve relevant documents"""
        start = time.time()
        q_emb = self.embedder.encode([query], normalize_embeddings=True)
        D, I = self.index.search(q_emb, top_k)
        retrieved_docs = [documents[i] for i in I[0]]
        context = "\n\n".join(retrieved_docs)
        retrieval_time = time.time() - start
        return context, I[0], D[0], retrieval_time

    def generate(self, query, context, temperature=0.7, top_p=0.95, top_k=50, max_new_tokens=512):
        """Generate answer using LLM"""
        if self.generator is None:
            return "Model not loaded", 0.0

        prompt = f"""Based on the following Amazon product information, answer the user's question accurately and naturally.

Context:
{context}

Question: {query}

Answer:"""

        start = time.time()
        try:
            output = self.generator(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                pad_token_id=self.generator.tokenizer.eos_token_id,
            )[0]["generated_text"]

            # Extract answer
            if "Answer:" in output:
                answer = output.split("Answer:")[-1].strip()
            else:
                answer = output[len(prompt) :].strip()
        except Exception as e:
            print(f"Generation error: {e}")
            answer = "Generation failed"

        generation_time = time.time() - start
        return answer, generation_time


# ================================
# Step 6: Grid Search Experiment Runner
# ================================


def run_grid_search():
    """Run comprehensive grid search experiment"""

    results = []
    experiment_id = 0

    # Generate all combinations
    embedding_models = EXPERIMENT_CONFIG["embedding_models"]
    generation_models = EXPERIMENT_CONFIG["generation_models"]

    total_experiments = (
        len(embedding_models)
        * len(generation_models)
        * len(EXPERIMENT_CONFIG["top_k"])
        * len(EXPERIMENT_CONFIG["temperature"])
        * len(EXPERIMENT_CONFIG["top_p"])
        * len(EXPERIMENT_CONFIG["top_k_sampling"])
        * len(EXPERIMENT_CONFIG["max_new_tokens"])
        * len(TEST_QUERIES)
    )

    print(f"\n{'='*60}")
    print(f"STARTING GRID SEARCH: {total_experiments} total experiments")
    print(f"{'='*60}\n")

    # Iterate through embedding models
    for emb_model in embedding_models:
        # Iterate through generation models
        for gen_model in generation_models:

            print(f"\n{'='*60}")
            print(f"Testing: {emb_model} + {gen_model}")
            print(f"{'='*60}")

            # Initialize RAG system
            rag = ConfigurableRAG(emb_model, gen_model)
            rag.setup_embedder()
            rag.setup_generator()

            if rag.generator is None:
                print(f"Skipping {gen_model} due to loading error")
                continue

            # Iterate through all hyperparameter combinations
            for top_k, temp, top_p, top_k_samp, max_tokens in product(
                EXPERIMENT_CONFIG["top_k"],
                EXPERIMENT_CONFIG["temperature"],
                EXPERIMENT_CONFIG["top_p"],
                EXPERIMENT_CONFIG["top_k_sampling"],
                EXPERIMENT_CONFIG["max_new_tokens"],
            ):
                # Test on all queries
                for query_data in TEST_QUERIES:
                    experiment_id += 1

                    query = query_data["query"]
                    reference = query_data["reference"]

                    print(f"\nExperiment {experiment_id}/{total_experiments}")
                    print(f"Query: {query[:60]}...")
                    print(
                        f"Params: k={top_k}, temp={temp}, top_p={top_p}, "
                        f"top_k={top_k_samp}, max_tokens={max_tokens}"
                    )

                    try:
                        # Retrieve
                        context, doc_ids, scores, ret_time = rag.retrieve(query, top_k)

                        # Generate
                        answer, gen_time = rag.generate(
                            query, context, temp, top_p, top_k_samp, max_tokens
                        )

                        # Calculate all metrics
                        eval_start = time.time()
                        all_metrics = metrics_calculator.calculate_all_metrics(
                            answer, reference, context
                        )
                        composite_score = metrics_calculator.calculate_composite_score(all_metrics)
                        eval_time = time.time() - eval_start

                        # Store results
                        result = {
                            "experiment_id": experiment_id,
                            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "query": query,
                            "reference": reference,
                            "answer": answer[:300],  # Truncate for CSV
                            "embedding_model": emb_model,
                            "generation_model": gen_model,
                            "top_k": top_k,
                            "temperature": temp,
                            "top_p": top_p,
                            "top_k_sampling": top_k_samp,
                            "max_new_tokens": max_tokens,
                            # All metrics
                            "rouge_1_f1": all_metrics["rouge_1_f1"],
                            "rouge_1_precision": all_metrics["rouge_1_precision"],
                            "rouge_1_recall": all_metrics["rouge_1_recall"],
                            "rouge_l_f1": all_metrics["rouge_l_f1"],
                            "rouge_l_precision": all_metrics["rouge_l_precision"],
                            "rouge_l_recall": all_metrics["rouge_l_recall"],
                            "bleu": all_metrics["bleu"],
                            "meteor": all_metrics["meteor"],
                            "bert_score_f1": all_metrics["bert_score_f1"],
                            "bert_score_precision": all_metrics["bert_score_precision"],
                            "bert_score_recall": all_metrics["bert_score_recall"],
                            "embedding_similarity": all_metrics["embedding_similarity"],
                            "faithfulness": all_metrics["faithfulness"],
                            "composite_score": composite_score,
                            # Performance metrics
                            "answer_length": len(answer),
                            "retrieval_time": ret_time,
                            "generation_time": gen_time,
                            "evaluation_time": eval_time,
                            "total_time": ret_time + gen_time + eval_time,
                            "avg_retrieval_score": float(np.mean(scores)),
                        }

                        results.append(result)

                        print(
                            f"✓ Composite Score: {composite_score:.3f} | "
                            f"ROUGE-1: {all_metrics['rouge_1_f1']:.3f} | "
                            f"BERTScore: {all_metrics['bert_score_f1']:.3f} | "
                            f"Time: {ret_time + gen_time:.2f}s"
                        )

                    except Exception as e:
                        print(f"✗ Error: {e}")
                        continue

            # Clean up to save memory
            del rag
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return pd.DataFrame(results)


# ================================
# Step 7: Run Experiments and Save Results
# ================================

print("Starting grid search experiments with advanced metrics...")
results_df = run_grid_search()

# Save results
output_file = "rag_grid_search_results_advanced.csv"
results_df.to_csv(output_file, index=False)
print(f"\n{'='*60}")
print(f"Results saved to: {output_file}")
print(f"{'='*60}")

# ================================
# Step 8: Comprehensive Analysis
# ================================

print("\n" + "=" * 60)
print("ANALYSIS: TOP 10 CONFIGURATIONS BY COMPOSITE SCORE")
print("=" * 60)

top_10 = results_df.nlargest(10, "composite_score")
print(
    top_10[
        [
            "experiment_id",
            "embedding_model",
            "generation_model",
            "top_k",
            "temperature",
            "composite_score",
            "rouge_1_f1",
            "bert_score_f1",
            "meteor",
        ]
    ].to_string()
)

# Best overall configuration
best_config = results_df.loc[results_df["composite_score"].idxmax()]

print("\n" + "=" * 60)
print("🏆 BEST CONFIGURATION FOUND")
print("=" * 60)
print(f"\nModel Configuration:")
print(f"  Embedding Model: {best_config['embedding_model']}")
print(f"  Generation Model: {best_config['generation_model']}")
print(f"\nHyperparameters:")
print(f"  Top K: {best_config['top_k']}")
print(f"  Temperature: {best_config['temperature']}")
print(f"  Top P: {best_config['top_p']}")
print(f"  Top K Sampling: {best_config['top_k_sampling']}")
print(f"  Max New Tokens: {best_config['max_new_tokens']}")
print(f"\nPerformance Metrics:")
print(f"  Composite Score: {best_config['composite_score']:.4f}")
print(f"  ROUGE-1 F1: {best_config['rouge_1_f1']:.4f}")
print(f"  ROUGE-L F1: {best_config['rouge_l_f1']:.4f}")
print(f"  BLEU: {best_config['bleu']:.4f}")
print(f"  METEOR: {best_config['meteor']:.4f}")
print(f"  BERTScore F1: {best_config['bert_score_f1']:.4f}")
print(f"  Embedding Similarity: {best_config['embedding_similarity']:.4f}")
print(f"  Faithfulness: {best_config['faithfulness']:.4f}")
print(f"\nTiming:")
print(f"  Retrieval Time: {best_config['retrieval_time']:.2f}s")
print(f"  Generation Time: {best_config['generation_time']:.2f}s")
print(f"  Total Time: {best_config['total_time']:.2f}s")
print("=" * 60)

# Detailed metric analysis
print("\n" + "=" * 60)
print("METRIC CORRELATIONS")
print("=" * 60)

metric_cols = [
    "rouge_1_f1",
    "rouge_l_f1",
    "bleu",
    "meteor",
    "bert_score_f1",
    "embedding_similarity",
    "faithfulness",
]
correlations = results_df[metric_cols].corr()
print(correlations.round(3))

# Summary statistics by model
print("\n" + "=" * 60)
print("AVERAGE PERFORMANCE BY MODEL COMBINATION")
print("=" * 60)

model_summary = (
    results_df.groupby(["embedding_model", "generation_model"])
    .agg(
        {
            "composite_score": ["mean", "std", "max"],
            "bert_score_f1": "mean",
            "meteor": "mean",
            "embedding_similarity": "mean",
            "total_time": "mean",
        }
    )
    .round(4)
    .sort_values(("composite_score", "mean"), ascending=False)
)

print(model_summary)

# Save summaries
model_summary.to_csv("rag_model_summary_advanced.csv")
results_df.describe().to_csv("rag_statistics_summary.csv")

print("\n" + "=" * 60)
print("FILES GENERATED:")
print("=" * 60)
print(f"1. {output_file} - All experiment results")
print("2. rag_model_summary_advanced.csv - Model performance summary")
print("3. rag_statistics_summary.csv - Statistical summary")
print("=" * 60)

# Metric weight contribution analysis
print("\n" + "=" * 60)
print("METRIC WEIGHT CONTRIBUTION TO COMPOSITE SCORE")
print("=" * 60)
for metric, weight in METRIC_WEIGHTS.items():
    print(f"{metric:.<30} {weight:.2%}")
print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

2025-12-05 05:51:10.411168: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764913870.754041      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764913870.847971      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

✓ Composite Score: 0.006 | ROUGE-1: 0.000 | BERTScore: 0.000 | Time: 0.63s

Experiment 575/576
Query: What is the best iPhone lightning cable in the list?...
Params: k=5, temp=0.7, top_p=0.95, top_k=50, max_tokens=512


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Composite Score: 0.013 | ROUGE-1: 0.000 | BERTScore: 0.000 | Time: 0.51s

Experiment 576/576
Query: Suggest me some good long lasting headphones...
Params: k=5, temp=0.7, top_p=0.95, top_k=50, max_tokens=512


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Composite Score: 0.002 | ROUGE-1: 0.000 | BERTScore: 0.000 | Time: 0.73s

Results saved to: rag_grid_search_results_advanced.csv

ANALYSIS: TOP 10 CONFIGURATIONS BY COMPOSITE SCORE
     experiment_id         embedding_model                    generation_model  top_k  temperature  composite_score  rouge_1_f1  bert_score_f1    meteor
121            122  BAAI/bge-small-en-v1.5  mistralai/Mistral-7B-Instruct-v0.2      5          0.7         0.603812    0.362069       0.887094  0.370402
385            386   BAAI/bge-base-en-v1.5            Qwen/Qwen2.5-7B-Instruct      3          0.3         0.599015    0.359375       0.890503  0.404348
29              30  BAAI/bge-small-en-v1.5            Qwen/Qwen2.5-7B-Instruct      3          0.7         0.596234    0.384615       0.904481  0.289868
441            442   BAAI/bge-base-en-v1.5            Qwen/Qwen2.5-7B-Instruct      5          0.7         0.593093    0.350365       0.883127  0.328044
421            422   BAAI/bge-base-en-v1.5          